In [1]:
!pip install pytorch-forecasting --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
  Using cached pytorch_forecasting-1.7.0-py3-none-any.whl.metadata (14 kB)
  Using cached lightning-2.6.1-py3-none-any.whl.metadata (44 kB)
  Using cached scikit_base-0.13.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached lightning_utilities-0.15.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached pytorch_lightning-2.6.1-py3-none-any.whl.metadata (21 kB)
  Using cached aiohttp-3.13.5-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (20 kB)
  Using cached multidict-6.7.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2

In [8]:
import numpy as np
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import Baseline
from pytorch_forecasting.metrics.point import MAE

In [9]:
class ZillowHousingDataService:
    
    def __init__(self, new_construction_price_per_square_foot_data_set_empty={}):
        self.new_construction_price_per_square_foot_data_set_formatted = new_construction_price_per_square_foot_data_set_empty
        
    def get_new_construction_price_per_square_foot_data_set_formatted(self):
        return self.new_construction_price_per_square_foot_data_set_formatted
        
    def get_new_construction_price_per_square_foot_data_set_training(self, region_id):#, indices):
        return self.new_construction_price_per_square_foot_data_set_formatted[region_id]#.take(indices)
        
    def format_new_construction_price_per_square_foot_data_set(self, create_repeating_array_function, data_frames, region_ids=None):
        
        if region_ids == None:
            index_values = data_frames.index
            
        else:
            index_values = region_ids
            
        for index_value in index_values:
            
            new_data_frame = data_frames.loc[[index_value]]
            
            time_series_data = new_data_frame.get(data_frames.columns[4:len(data_frames.columns)])
            
            new_data_frame.drop(columns=new_data_frame.columns, inplace=True)
            
            new_data_frame = new_data_frame.assign(RegionID=[create_repeating_array_function(index_value, len(time_series_data.columns))])
            
            new_data_frame = new_data_frame.explode("RegionID")
            
            new_data_frame.reset_index(drop=True, inplace=True)
            
            new_data_frame.insert(loc=0, column="Index", value=new_data_frame.index, allow_duplicates=False)
            
            new_data_frame = new_data_frame.assign(Date=time_series_data.columns.T)
            
            new_data_frame = new_data_frame.assign(Price=time_series_data.T.to_numpy(copy=True))
            
            self.new_construction_price_per_square_foot_data_set_formatted[index_value] = new_data_frame

In [10]:
data_file_path = "~/ece492-final-project/"
data_file_name = "Metro_new_con_median_sale_price_per_sqft_uc_sfrcondo_month.csv"

data_frames = pd.read_csv(data_file_path + data_file_name)
data_frames.set_index("RegionID", inplace=True)
data_frames.sort_index(inplace=True)

In [11]:
data_service = ZillowHousingDataService()
data_service.format_new_construction_price_per_square_foot_data_set(np.repeat, data_frames) #data_frames.index[0:1:1]
formatted_data_sets = data_service.get_new_construction_price_per_square_foot_data_set_formatted()

In [12]:
data = formatted_data_sets[102001]

training = TimeSeriesDataSet(
    data=data,
    time_idx="Index",
    target="Price",
    group_ids=["RegionID"]
)

In [13]:
batch_size = 128
validation = TimeSeriesDataSet.from_dataset(training, data, min_prediction_idx=training.index.time.max() + 1, stop_randomization=True)
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=2)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=2)

In [29]:
baseline_predictions = Baseline().predict(val_dataloader, return_y=True)

metric = MAE()
model = Baseline()
for i, x in enumerate(baseline_predictions):
    print(x)
    print(baseline_predictions[i])
    #metric.update(model(x))
    break
    
#metric.compute()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


tensor([[206.4516],
        [208.0936],
        [208.9493],
        [209.4857],
        [207.8303],
        [208.0355],
        [203.7493],
        [203.7133],
        [205.8468],
        [207.6758],
        [208.2962],
        [207.9566],
        [207.7066],
        [207.2335],
        [206.2226],
        [208.3165],
        [206.8965],
        [206.6667],
        [204.7318],
        [204.4482],
        [204.3085],
        [205.1917],
        [203.6263],
        [203.9348],
        [203.7976],
        [204.7514],
        [204.5455],
        [206.6984],
        [203.4759],
        [201.6942]])
tensor([[206.4516],
        [208.0936],
        [208.9493],
        [209.4857],
        [207.8303],
        [208.0355],
        [203.7493],
        [203.7133],
        [205.8468],
        [207.6758],
        [208.2962],
        [207.9566],
        [207.7066],
        [207.2335],
        [206.2226],
        [208.3165],
        [206.8965],
        [206.6667],
        [204.7318],
        [204.4482],